# Plenary 8 · Six questions, six numbers

Workshop **Data Driven Domestic Revenue Mobilization**, Bangkok, October 2026.

Six worked-example cards, one per question: the data held, the query in plain words, a low-risk
action, one documented case with its number and source, what to record, what it unlocks. Three cards
are the tax side's and come from published cases; three are run as one query each on the synthetic
set. Then what the six have in common, the engine in plain words, your written pick, and what it takes.

**How to use it.** Press ▶ on each cell from top to bottom, or *Runtime › Run all*. Change only the sliders and drop-down menus. The code is folded: double-click a title to see it. Every result ends with a **Finding** sentence. Everything computed here is synthetic; every outside number is cited with its source.

In [ ]:
#@title 0 · Where the data comes from
#@markdown Leave these as they are. Your analysts can point DATA_URL at their own extract (section 6).
DATA_URL = "https://raw.githubusercontent.com/FrancoisChastel/CustomsWorkshopNotebooks/data-v8/data"  #@param {type:"string"}
INSTALL = True  #@param {type:"boolean"}

In [ ]:
#@title 0 · Setup (run me first)
import sys, subprocess, warnings
warnings.filterwarnings("ignore")
if INSTALL and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/FrancoisChastel/CustomsWorkshopNotebooks@data-v8"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from workshop_methods import notebook as nb
from workshop_methods.data import load_json, load_tables
from workshop_methods import quick_queries as qq, signals
nb.use_style()
pd.set_option("display.max_colwidth", 40)
CALIBRATION = "random draw finds something on 1 to 8 lines in a hundred; this set is enriched on the 20 exercise lines only."
def calibration_line():
    """Printed under every table that shows a rate."""
    display(Markdown(f"<small><i>Calibration: {CALIBRATION}</i></small>"))
HOME_USE = ("home_use", "warehouse_out")  # the regimes where duty is paid: the lines a price check ranks
T = load_tables(DATA_URL, ['features_lines', 'declarations_h2', 'tariff', 'licences', 'importers', 'tax_registry', 'vat_returns', 'trader_features'])
print(f"Loaded {len(T)} tables from {DATA_URL}. Ready.")

In [ ]:
#@title 0 · Prepare the tables
f, d2, tariff = T["features_lines"], T["declarations_h2"], T["tariff"]
importers, registry, returns, licences = T["importers"], T["tax_registry"], T["vat_returns"], T["licences"]
months = sorted(d2["month"].unique())
periods = sorted(returns["period"].unique())
last12 = f[(f["period_month"] > 24) & f["regime"].isin(HOME_USE)]
print(f"Months 19-36 of declarations ({months[0]} to {months[-1]}); the last 12 months hold {len(last12):,} lines that pay duty. "
      f"Returns for {returns['tin'].nunique():,} registered taxpayers over {len(periods)} periods.")

## 1 · Six worked-example cards

Each card is one query and one number. Under every table that shows a rate stands the calibration
line: this set is synthetic and enriched only where the data dictionary says so.

### Card 1 · Non-filers *(tax side, from published cases)*

| | |
| --- | --- |
| **Data held** | the VAT register and the returns filed by period; the customs import value per importer |
| **Query in plain words** | registered importers that missed returns in the last six periods, largest importers first |
| **Low-risk action** | a letter or a message that names the import value, before any assessment |
| **Documented case** | UK 2012 trial, 119,527 late payers; payment within 23 days 33.6% in the control group; best wording +5.0 points; £9.1 million extra in 23 days; marginal cost "practically zero" (Trial 1: 101,471 people, +5.1 points, £3.2 million). *Hallsworth, List, Metcalfe, Vlaev, NBER WP 20007, pp. 5, 11, 17, 19, 24–26.* Madagascar SMS: 15,885 late filers randomised; filing 7.2% → 9.8%; about US$375,000; US$329 per dollar spent. *World Bank Governance blog, 23 July 2019 (experiment year not stated).* IRS high-income non-filers: 135,270 cases monitored; of 111,566 in scope, 30,149 (27%) filed or assessed and 81,417 (73%) unworked; potential tax about $15.7 billion; two-year IT pause; 4,918 notices to people who had filed; no programme-level tracking. *TIGTA 2026-308-047, pp. 6–7, 9, 14, 18 (the 73% is of 111,566, not 135,270).* |
| **What to record** | letter sent, return filed within 23 days, amount paid |
| **What it unlocks** | the cheapest signal the tax side holds, with the customs number in the letter |

In [ ]:
#@title Card 1 · Registered importers with missing returns
MISSING = 2  #@param {type:"slider", min:1, max:6, step:1}
nb.steps("Which registered importers missed returns, and how large are they?",
         "Take the last six periods of returns; count the unfiled periods per taxpayer.",
         f"Keep taxpayers with {MISSING} or more unfiled; join them to the importers by tax ID and sort by 12-month import value.")
listed = qq.non_filers(returns, importers, d2)
listed = listed[listed["missing_returns"] >= MISSING]
nb.show(listed.head(8).set_index("importer_id")[["tin", "missing_returns", "import_value_12m"]], formats={"import_value_12m": "{:,.0f}"})
calibration_line()
registered = importers["tin"].fillna("").ne("").sum()
nb.finding(f"{len(listed)} registered importers ({len(listed) / registered:.0%} of those with a tax ID) miss {MISSING} or more of the last six "
           f"returns; together they imported LCU {listed['import_value_12m'].sum():,.0f} in 12 months. The letter that names the import "
           "value is the lowest-risk action in the book; record the return filed within 23 days.")

### Card 2 · Refund risk *(tax side, from published cases; no synthetic query: this set holds no refund claims)*

| | |
| --- | --- |
| **Data held** | refund claims, the audits they trigger, their outcomes; customs-validated exports |
| **Query in plain words** | the share of audits triggered by refunds, and their hit rate against planned audits |
| **Low-risk action** | pre-fill the export figure from customs into the return, so the claim starts from a validated number |
| **Documented case** | Moldova: refund-triggered audits 26.6% (2018) → 48.9% (2024) of audits; hit rate 10.4% vs 92.4% for planned audits; 30% planned / 70% refund and insolvency. *IMF, Republic of Moldova, Strengthening Tax Compliance Management, February 2026, pp. 14, 31.* |
| **What to record** | claims pre-filled, claims adjusted after pre-filling, audit hit rate by trigger |
| **What it unlocks** | Kenya: customs-validated exports pre-filled into the VAT return under zero-rated supplies; figures can no longer be adjusted at filing; controls in place from May 2026. *KPMG East Africa tax alert, May 2026.* |

### Card 3 · Arrears *(tax side, from published cases; the cell prints the ADB figures only)*

| | |
| --- | --- |
| **Data held** | the debt ledger: arrears by age, the share deemed uncollectible; the importers among the debtors |
| **Query in plain words** | the arrears ratio, and how much of the debt the administration itself deems uncollectible |
| **Low-risk action** | record the ratio and the uncollectible share every quarter; the clearance hold on importing debtors is not claimed here, because no source gives a result for it |
| **Documented case** | Asia-Pacific: over 45% of reporting economies deemed more than 40% of debt uncollectible; about 25% deemed 60% or more; arrears ratios under 5% to over 200%. *ADB Governance Brief 49, Chooi, 2023, pp. 2–3 and Figure 3.* |
| **What to record** | arrears ratio, share deemed uncollectible, age of the debt |
| **What it unlocks** | a debt book that customs can read before release, once the identifier is shared |

In [ ]:
#@title Card 3 · The arrears figures, from the published case
nb.steps("What do published cases say about arrears in the region?", "Read the ADB figures; this set holds no debt ledger, so nothing is computed.")
adb = pd.DataFrame({"figure": ["reporting economies deeming more than 40% of debt uncollectible", "reporting economies deeming 60% or more uncollectible",
                               "arrears ratios across the economies"],
                    "value": ["over 45%", "about 25%", "under 5% to over 200%"],
                    "source": ["ADB Governance Brief 49, Chooi, 2023, pp. 2-3 and Figure 3"] * 3}).set_index("figure")
nb.show(adb)
nb.finding("Over 45% of reporting economies deem more than 40% of their debt uncollectible (ADB Governance Brief 49): the two numbers to "
           "record are the arrears ratio and the share deemed uncollectible. No source gives a result for clearance holds on importing "
           "debtors, so none is claimed here.")

### Card 4 · Valuation *(customs)*

| | |
| --- | --- |
| **Data held** | declarations with value, weight and goods code; the peer median per code and origin |
| **Query in plain words** | the duty at stake when lines priced far below their peers are read at the typical value |
| **Low-risk action** | a valuation query through the procedure, with the reference shown; never a reassessment from the reference |
| **Documented case** | Madagascar: third-party valuation advice +21.7 points of fraud findings and +5.2 points of tax; detailed comments +3.1 and +1; monitoring tags more scanning, no more fraud; effect smaller where graft opportunities are large. *Chalendard et al., World Bank WPS 9254, 2020, abstract.* WCO: a national valuation database is a risk-assessment tool; values may not be used as the customs value, a substitute value or a minimum; a difference is "not by itself a reliable indicator of potential risk". *WCO Guidelines on the development and use of a national valuation database as a risk assessment tool.* |
| **What to record** | query raised, query upheld, duty recovered, per importer |
| **What it unlocks** | a reference beside every line; the ladder of session 1B |

In [ ]:
#@title Card 4 · Duty at stake in lines priced far below their peers
THRESHOLD = 0.70  #@param {type:"slider", min:0.5, max:0.9, step:0.05}
nb.steps("How much duty would the last 12 months carry if low-priced lines were read at the typical value?",
         "Drop lines whose quantity, unit and weight disagree (a units error is not a price).",
         f"Keep home-use lines below {THRESHOLD:.0%} of their peer median (the precomputed median for the same code and origin, with its fallback).",
         "Duty at stake = (peer median - declared value per kilo) x weight x duty rate, summed by importer.")
clean = last12[signals.units_consistent(last12, tariff)]
low = clean[(clean["share_of_median"] < THRESHOLD) & (clean["peer_level_used"] > 0)]
stake = ((low["peer_median_used"] - low["value_per_kg"]) * low["net_kg"] * low["duty_rate"] / 100).rename("duty_at_stake")
display(Markdown(f"### Duty at stake, last 12 months: LCU {stake.sum():,.0f}"))
top = pd.DataFrame({"duty_at_stake": stake, "importer_id": low["importer_id"], "share": low["share_of_median"]}).groupby("importer_id").agg(
    duty_at_stake=("duty_at_stake", "sum"), lines=("duty_at_stake", "size"), median_share_of_peers=("share", "median")).nlargest(10, "duty_at_stake")
top["share_of_total"] = top["duty_at_stake"] / stake.sum()
nb.show(top, formats={"duty_at_stake": "{:,.0f}", "median_share_of_peers": "{:.2f}", "share_of_total": "{:.0%}"})
calibration_line()
nb.finding(f"LCU {stake.sum():,.0f} of duty is at stake on {len(low):,} lines from {low['importer_id'].nunique()} importers; the ten largest "
           f"carry {top['share_of_total'].sum():.0%} of it. The reference is what lifts findings (Madagascar: +21.7 points), and it is a "
           "reason to ask, never a value to assess: record query raised, upheld and duty recovered per importer.")

In [ ]:
#@title Card 4 · By chapter
by_chapter = stake.groupby(low["chapter"]).sum().sort_values()
fig, ax = nb.chart(f"Chapter {by_chapter.index[-1]} holds the largest share of duty at stake")
ax.barh(by_chapter.index, by_chapter.values / 1e6, color=[nb.COLORS["amber"] if c == by_chapter.index[-1] else nb.COLORS["accent"] for c in by_chapter.index])
ax.set_xlabel("duty at stake, LCU million"); plt.show()
nb.finding(f"Chapter {by_chapter.index[-1]} holds {by_chapter.iloc[-1] / by_chapter.sum():.0%} of the duty at stake and chapter "
           f"{by_chapter.index[-2]} {by_chapter.iloc[-2] / by_chapter.sum():.0%}: the post-clearance audit plan starts there.")

### Card 5 · Exemptions *(customs)*

| | |
| --- | --- |
| **Data held** | lines entered under a relief scheme; the licences with their limits |
| **Query in plain words** | the duty foregone by scheme, and the beneficiaries that used more than their licence allows |
| **Low-risk action** | reconcile licences with clearances monthly; suspend the relief at the limit |
| **Documented case** | Ghana 2023: import exemptions GH₵3,545 million of GH₵4,619 million tax expenditure (76.76%), 0.44% of GDP; parliamentary exemptions 48%. *Ministry of Finance, Ghana, 2023 Tax Expenditure Report, Table 1 and section 7.* Uganda FY2022/23: total tax expenditure UGX 2,972 billion, 1.62% of GDP, 12.5% of collections; customs UGX 794 billion, 0.43% of GDP; stays of application and remissions larger than named exemptions. *MoFPED Uganda, Tax Expenditure Report FY2022/23.* No source gives a duration for the reconciliation, so none is claimed. |
| **What to record** | relief used against the limit, licence suspended, duty recovered |
| **What it unlocks** | a tax-expenditure number the ministry can publish |

In [ ]:
#@title Card 5 · Duty foregone by scheme, and beneficiaries above their licence
nb.steps("What does relief cost, and who uses more than allowed?",
         "Duty foregone = value of the lines entered under a relief scheme x the normal (MFN) duty rate, by scheme, months 19-36.",
         "For each beneficiary: the relieved value over the last 12 months divided by its licence limit.")
foregone = qq.foregone_duty(d2, tariff)
use = qq.relief_overuse(d2, licences)
view = use.head(6).set_index("importer_id")[["scheme", "relief_value_12m", "licence_limit_lcu", "use_of_limit"]]
fig, ax = nb.chart(f"Scheme {foregone.index[0]} accounts for {foregone.iloc[0] / foregone.sum():.0%} of the duty foregone")
ax.barh(foregone.index[::-1], foregone.values[::-1] / 1e6, color=nb.COLORS["accent"]); ax.set_xlabel("duty foregone, months 19-36, LCU million"); plt.show()
nb.show(view, formats={"relief_value_12m": "{:,.0f}", "licence_limit_lcu": "{:,.0f}", "use_of_limit": "{:.2f}x"}, highlight=view["use_of_limit"] > 1)
over = view[view["use_of_limit"] > 1]
nb.finding(f"Relief cost LCU {foregone.sum():,.0f} of duty over months 19-36, {foregone.iloc[0] / foregone.sum():.0%} of it under scheme "
           f"{foregone.index[0]}. {len(over)} beneficiaries used more than their licence allows ({', '.join(over.index)}, at "
           f"{over['use_of_limit'].min():.2f} to {over['use_of_limit'].max():.2f} times the limit): relief overuse is foregone duty with "
           "a signature on it.")

### Card 6 · Import VAT *(customs and tax together)*

| | |
| --- | --- |
| **Data held** | import VAT paid at the border per importer; the VAT register; the credit claimed on returns |
| **Query in plain words** | importers above the VAT threshold the register does not hold; and importers whose credit on returns exceeds the import VAT they paid |
| **Low-risk action** | the list to the registration unit with the import values; the border payments shared with the VAT office monthly |
| **Documented case** | Denmark: VAT DKK 228 billion (2020), 520,000 registered businesses; mismatch DKK 261 billion a year 2014–2020, DKK 100 billion high-risk; about 3% of EU traders checked; VIES used in 7% of 71 reviewed cases; single differences above DKK 1 billion. *Rigsrevisionen report 3/2021, pp. 1–4.* |
| **What to record** | registered after referral, VAT assessed, credit corrected |
| **What it unlocks** | Indonesia: joint supervision list of 4,301 taxpayers, 3,881 followed up in 2025, Rp3.1 trillion; 2024 Rp1.27 trillion; legal basis KMK 210/KMK.01/2021 as amended by KMK-570/KM.1/2023. *DDTC News, 2025 and 2026 reports.* |

In [ ]:
#@title Card 6 · Large importers the VAT register does not hold
VAT_THRESHOLD = 1_500_000  #@param {type:"integer"}
nb.steps("Which importers above the VAT threshold are not registered for VAT?",
         "Sum each importer's home-use imports and the import VAT it paid over the last 12 months.",
         "Look up the tax ID customs holds in the registry; keep importers not found, or found but not VAT-registered, above the threshold.")
list1, list2 = qq.import_vat_lists(d2, importers, registry, returns, VAT_THRESHOLD)
view = list1.head(8)[["customs_tin", "import_value_12m", "import_vat_paid"]]
nb.show(view, formats={"import_value_12m": "{:,.0f}", "import_vat_paid": "{:,.0f}"})
calibration_line()
blank = int(list1["customs_tin"].eq("").sum())
nb.finding(f"{len(list1)} importers above the threshold are not in the VAT register or not registered for VAT ({blank} of them with no tax ID at "
           f"all on the customs side); they imported LCU {list1['import_value_12m'].sum():,.0f} in 12 months and should be charging VAT on "
           "their sales. Send the list to the registration unit with the import values.")

In [ ]:
#@title Card 6 · Import VAT claimed on returns against import VAT paid at the border
nb.steps("Does anyone claim more import VAT on its returns than it paid at the border?",
         "Sum the import VAT paid at the border and the import VAT credit claimed on returns over 12 months, per tax ID.",
         "Largest difference first.")
view = list2.head(6)[["import_vat_paid", "import_vat_credit", "credit_minus_paid", "credit_to_paid"]]
nb.show(view, formats={"import_vat_paid": "{:,.0f}", "import_vat_credit": "{:,.0f}", "credit_minus_paid": "{:,.0f}", "credit_to_paid": "{:.1f}x"})
gap = list2[list2["credit_to_paid"] > 1.5]
nb.finding(f"{len(gap)} importers claim more than 1.5 times the import VAT they paid at the border; {view.index[0]} and {view.index[1]} claim "
           f"{view['credit_to_paid'].iloc[0]:.1f} and {view['credit_to_paid'].iloc[1]:.1f} times, LCU {gap['credit_minus_paid'].sum():,.0f} "
           "between them all: a reconciliation customs and tax can only do together, monthly.")

Three things are deliberately not claimed because no source exists: a result for clearance holds on
importing debtors, a duration for exemption reconciliation, a recovered-revenue figure for a batch
unit-value desk review.

## 2 · What the six have in common

In [ ]:
#@title 2 · One extract, one number, one low-risk action, one outcome field
common = pd.DataFrame({
    "the extract": ["returns by period, imports by importer", "refund claims and their audits", "the debt ledger", "declarations with the peer median",
                    "relief lines and licences", "border VAT, the register, the returns"],
    "the number": ["importers with missing returns", "hit rate of refund audits", "share deemed uncollectible", "duty at stake",
                   "duty foregone; use of the limit", "unregistered importers; credit above payment"],
    "who acts": ["tax", "tax", "tax", "customs", "customs", "both"],
    "the outcome field": ["return filed within 23 days", "claim adjusted after pre-filling", "debt collected or written off", "query upheld, duty recovered",
                          "licence suspended, duty recovered", "registered; credit corrected"],
    "what it unlocks": ["the customs number in the letter", "exports pre-filled from customs", "a debt book customs reads before release",
                        "a reference beside every line", "a tax-expenditure number", "a joint list with a legal basis"],
}, index=["non-filers", "refund risk", "arrears", "valuation", "exemptions", "import VAT"])
nb.show(common)
nb.finding("Every card is one extract the administration already holds, one number, one action an officer can take without a new law, "
           "and one outcome field; the fourth column is what the next quarter's ranking learns from. Three of the six unlock only when "
           "the two administrations share an identifier and a return flow.")

## 3 · The engine in plain words

Behind the six cards is one loop: **extract** the data you hold, **size** the question with one number,
**rank** who comes first, **act** with the lowest-risk step, **measure** the outcome and write it back,
**learn** by ranking again next quarter with the outcomes in. No model is needed for the first turn of
the loop; the outcome field is.

In [ ]:
#@title 3 · The loop, for the three customs cards
loop = pd.DataFrame({
    "valuation": ["declarations, tariff, the peer median", "duty at stake", "importers by duty at stake", "valuation query through the procedure",
                  "query raised, upheld, duty recovered", "the reference moves with the market; the rule carries a date"],
    "exemptions": ["relief lines, licences", "duty foregone; use of the limit", "beneficiaries above their limit", "licence review",
                   "licence suspended, duty recovered", "the limit check runs monthly"],
    "import VAT": ["border VAT, the register, the returns", "unregistered importers; credit above payment", "largest first",
                   "referral to registration; reconciliation with tax", "registered, VAT assessed, credit corrected", "a shared identifier and a return flow"],
}, index=["extract", "size", "rank", "act", "measure", "learn"])
nb.show(loop)
nb.finding("Each question needs one extract and creates one outcome field; without the outcome field the next ranking cannot learn which "
           "actions paid, and the engine stays a report.")

## 4 · Your written pick: a one-pager

Three minutes in silence. Fill the five boxes (the cell prints a page you can print or copy), then the
sheets are collected. Box 3 asks for a unit and a name, and how many of the list you can work this
quarter; box 4 asks how you will know it worked.

In [ ]:
#@title 4 · Print the pick sheet
QUESTION = ""  #@param ["", "non-filers", "refund risk", "arrears", "valuation", "exemptions", "import VAT"]
from IPython.display import HTML
boxes = [
    ("1 · The question I pick", "one of the six, and the number it starts from on our data"),
    ("2 · The data I hold and the query in plain words", "the tables, the field that joins them, the one sentence the query says"),
    ("3 · Owner and capacity", "a unit and a name; how many of the list you can work this quarter"),
    ("4 · Baseline, comparison group and window", "the number before we start; who is left alone so the difference means something; the weeks we measure"),
    ("5 · What I will record, and when I report", "the outcome field, who writes it, the date the number is read out"),
]
style = ("<style>.pick{font-family:Arial,sans-serif;color:#3C4A5E;max-width:900px}.pick h2{color:#0E2F5A;margin:0 0 4px}"
         ".pick .box{border:1px solid #D9E0EA;border-radius:6px;padding:10px 14px;margin:10px 0;min-height:88px;background:#FDFDFC}"
         ".pick .box b{color:#0E2F5A}.pick .box small{color:#6A7686}.pick .foot{font-size:11px;color:#6A7686}"
         "@media print{.pick .box{min-height:120px;page-break-inside:avoid}}</style>")
html = style + "<div class='pick'><h2>My written pick</h2><small>Plenary 8 · one question, one number, one owner</small>"
html += "".join(f"<div class='box'><b>{title}</b>" + (f" &nbsp; <i>{QUESTION}</i>" if i == 0 and QUESTION else "") +
                f"<br><small>{hint}</small></div>" for i, (title, hint) in enumerate(boxes))
html += "<div class='foot'>IMF · Data and AI for Customs</div></div>"
display(HTML(html))
nb.finding("A pick with a named owner, a capacity and a comparison group is a plan; without box 3 and box 4 it is a wish.")

## 5 · What it takes

The six queries run on data the administration already holds. What decides whether they run next
month too is where the data lives and who looks after it.

In [ ]:
#@title 5 · Where data can live: four places, and what each lets you do
house = pd.DataFrame({
    "what it is": ["the live system officers clear goods in", "a copy refreshed every night that nobody writes to",
                   "the replica plus the register, the returns and the outcomes, cleaned and joined on one identifier",
                   "everything as it arrives, documents and images and partner statistics, beside the warehouse"],
    "what it lets you do": ["one extract at a time, by request to IT", "every monthly query, without touching the live system",
                            "the six cards as standing reports; a score refreshed monthly; the random share drawn before the rules",
                            "reading documents and images; the learning system of session 1B"],
    "time to build": ["in place", "weeks", "months", "a year or more"],
}, index=["clearance system", "read-only replica", "warehouse", "lake"])
nb.show(house)
habits = ["One named owner per table, and a data dictionary kept beside the data.",
          "The six-field quality check read monthly as a number, not fixed once.",
          "Every control outcome written back to the line the same week.",
          "A random share drawn before the rules run, its size fixed by the committee.",
          "Every rule and every score carries a date and an expiry.",
          "Minimum fields, an access log and a legal basis for every exchange with the other administration."]
display(Markdown("**Six governance habits**\n\n" + "\n".join(f"{i}. {h}" for i, h in enumerate(habits, 1))))
nb.finding("The first two places are enough for every card in this plenary; the warehouse is what makes them standing reports, and the "
           "six habits are what keep the numbers true. Going further means the third tier of data: what the other administration holds.")

## 6 · For your analysts

The customs queries in pandas and in SQL, side by side, against the same files. The SQL runs in DuckDB
here and pastes into most warehouses once the column names are mapped to yours (`hs8`, `customs_value_lcu`,
`net_kg`, `duty_rate`, `tin`, `period`, `filed`). The valuation query uses the precomputed peer median of
`features_lines`; in your own extract compute it first as `MEDIAN(customs_value_lcu / net_kg)` per code and
origin over 12 months, with the fallback to the code and then the heading when a cell has fewer than 30 lines.

In [ ]:
#@title 6 · The queries in SQL (DuckDB) and in pandas
import duckdb
con = duckdb.connect()
for name in ["features_lines", "declarations_h2", "tariff", "licences", "importers", "tax_registry", "vat_returns"]:
    con.register(name, T[name])
non_filers_sql = f"""
    WITH recent AS (SELECT tin, COUNT(*) FILTER (WHERE filed = 0) AS missing_returns
                    FROM vat_returns WHERE period >= '{periods[-6]}' GROUP BY 1)
    SELECT i.importer_id, i.tin, r.missing_returns
    FROM importers i JOIN recent r USING (tin) WHERE r.missing_returns >= 2"""
valuation_sql = """
    SELECT f.importer_id, SUM((f.peer_median_used - f.value_per_kg) * f.net_kg * f.duty_rate / 100) AS duty_at_stake
    FROM features_lines f JOIN tariff t USING (hs8)
    WHERE f.regime IN ('home_use', 'warehouse_out') AND f.period_month > 24 AND f.peer_level_used > 0
      AND f.share_of_median < 0.70 AND f.unit = t.unit          -- the unit check; the weight-against-quantity check is in signals.units_consistent
    GROUP BY 1 ORDER BY 2 DESC"""
relief_sql = """
    SELECT d.exemption_code, ROUND(SUM(d.customs_value_lcu * t.duty_mfn / 100)) AS foregone
    FROM declarations_h2 d JOIN tariff t USING (hs8)
    WHERE d.regime = 'home_use_exempt' GROUP BY 1 ORDER BY 2 DESC"""
vat_sql = f"""
    WITH imports AS (SELECT importer_id, SUM(customs_value_lcu) AS import_value_12m
                     FROM declarations_h2 WHERE regime IN ('home_use', 'warehouse_out') AND month > '{months[-13]}' GROUP BY 1)
    SELECT i.importer_id, i.import_value_12m
    FROM imports i LEFT JOIN importers p USING (importer_id) LEFT JOIN tax_registry r ON r.tin = p.tin
    WHERE i.import_value_12m > {VAT_THRESHOLD} AND (r.tin IS NULL OR NOT r.vat_registered) ORDER BY 2 DESC"""
sql = {name: con.execute(q).df() for name, q in [("non_filers", non_filers_sql), ("valuation", valuation_sql), ("relief", relief_sql), ("vat", vat_sql)]}
same_units = last12[(last12["unit"] == last12["hs8"].map(tariff.set_index("hs8")["unit"])) & (last12["share_of_median"] < 0.70) & (last12["peer_level_used"] > 0)]
pandas_valuation = ((same_units["peer_median_used"] - same_units["value_per_kg"]) * same_units["net_kg"] * same_units["duty_rate"] / 100).sum()
agree = {"non_filers": len(sql["non_filers"]) == int((qq.non_filers(returns, importers, d2)["missing_returns"] >= 2).sum()),
         "valuation": abs(sql["valuation"]["duty_at_stake"].sum() - pandas_valuation) < 1,
         "relief": np.allclose(sql["relief"].set_index("exemption_code")["foregone"].reindex(foregone.index), foregone, rtol=1e-6),
         "vat": set(sql["vat"]["importer_id"]) == set(list1.index)}
for q in (non_filers_sql, valuation_sql, relief_sql, vat_sql):
    print(q.strip(), "\n")
nb.finding("The SQL and the pandas versions give the same numbers for all four queries." if all(agree.values())
           else f"The SQL and pandas versions differ on {', '.join(k for k, v in agree.items() if not v)}: check the column map.")